# Tarea 5

#### Nombre: Sofia Catalina Soto Ferrada
#### Rut: 22.261.514-3

In [1]:
# Importamos las bibliotecas que utilizaremos
import numpy as np
import pandas as pd
import os
import pickle

In [2]:
class CatalogoEstelar:

    def __init__(self, ruta_csv):
        self.datos = pd.read_csv(ruta_csv)
        self.limpiar_datos()

    def limpiar_datos(self):
        columnas = [
            "id", "nombre", "temperatura_K",
            "luminosidad_Lsol", "distancia_pc"
        ]

        datos = self.datos[columnas].copy()

        numericas = [
            "temperatura_K", "luminosidad_Lsol", "distancia_pc"
        ]

        for columna in numericas:
            datos[columna] = pd.to_numeric(
                datos[columna], errors="coerce"
            )

        datos = datos.dropna(subset=columnas)

        validos = (
            np.isfinite(datos["temperatura_K"]) &
            np.isfinite(datos["luminosidad_Lsol"]) &
            np.isfinite(datos["distancia_pc"]) &
            (datos["temperatura_K"] > 0) &
            (datos["luminosidad_Lsol"] > 0) &
            (datos["distancia_pc"] > 0)
        )

        datos = datos[validos]
        datos = datos.drop_duplicates(subset="id", keep="first")

        self.id = datos["id"].astype(str).to_numpy()
        self.nombre = datos["nombre"].astype(str).to_numpy()
        self.temperatura_K = datos["temperatura_K"].to_numpy(dtype=float)
        self.luminosidad_Lsol = datos["luminosidad_Lsol"].to_numpy(dtype=float)
        self.distancia_pc = datos["distancia_pc"].to_numpy(dtype=float)

        print("Limpieza completada.")
        print("Estrellas validas:", len(self.id))

    def resumen(self):
        indice_caliente = np.argmax(self.temperatura_K)
        indice_cercana = np.argmin(self.distancia_pc)

        print("Resumen del catalogo estelar")
        print("Cantidad de estrellas:", len(self.id))
        print("Temperatura media:", round(np.mean(self.temperatura_K), 2), "K")
        print("Temperatura maxima:", round(np.max(self.temperatura_K), 2), "K")
        print("Luminosidad mediana:", round(np.median(self.luminosidad_Lsol), 2), "Lsol")
        print("Distancia minima:", round(np.min(self.distancia_pc), 2), "pc")
        print("Distancia maxima:", round(np.max(self.distancia_pc), 2), "pc")
        print("Estrella mas caliente:", self.nombre[indice_caliente])
        print("Estrella mas cercana:", self.nombre[indice_cercana])

    def guardar_csv(self):
        os.makedirs("resultados", exist_ok=True)

        datos = pd.DataFrame({
            "id": self.id,
            "nombre": self.nombre,
            "temperatura_K": self.temperatura_K,
            "luminosidad_Lsol": self.luminosidad_Lsol,
            "distancia_pc": self.distancia_pc
        })

        datos.to_csv("resultados/catalogo_copia.csv", index=False)
        print("Catalogo guardado en CSV.")

    def guardar_npy(self):
        os.makedirs("resultados", exist_ok=True)
        np.save("resultados/temperaturas.npy", self.temperatura_K)
        print("Temperaturas guardadas en NPY.")

    def guardar_npz(self):
        os.makedirs("resultados", exist_ok=True)

        np.savez_compressed(
            "resultados/catalogo_completo.npz",
            id=self.id.astype(str),
            nombre=self.nombre.astype(str),
            temperatura_K=self.temperatura_K,
            luminosidad_Lsol=self.luminosidad_Lsol,
            distancia_pc=self.distancia_pc
        )

        print("Catalogo guardado en NPZ.")

    def guardar_pickle(self):
        os.makedirs("resultados", exist_ok=True)

        datos = {
            "id": self.id,
            "nombre": self.nombre,
            "temperatura_K": self.temperatura_K,
            "luminosidad_Lsol": self.luminosidad_Lsol,
            "distancia_pc": self.distancia_pc
        }

        with open("resultados/catalogo_completo.pkl", "wb") as archivo:
            pickle.dump(datos, archivo)

        print("Catalogo guardado en Pickle.")

    def verificar_archivos(self):
        csv = CatalogoEstelar("resultados/catalogo_copia.csv")

        csv_ok = (
            np.array_equal(csv.id, self.id) and
            np.array_equal(csv.nombre, self.nombre) and
            np.allclose(csv.temperatura_K, self.temperatura_K) and
            np.allclose(csv.luminosidad_Lsol, self.luminosidad_Lsol) and
            np.allclose(csv.distancia_pc, self.distancia_pc)
        )

        temperaturas = np.load("resultados/temperaturas.npy")
        npy_ok = np.allclose(temperaturas, self.temperatura_K)

        with np.load("resultados/catalogo_completo.npz") as npz:
            npz_ok = (
                np.array_equal(npz["id"], self.id) and
                np.array_equal(npz["nombre"], self.nombre) and
                np.allclose(npz["temperatura_K"], self.temperatura_K) and
                np.allclose(npz["luminosidad_Lsol"], self.luminosidad_Lsol) and
                np.allclose(npz["distancia_pc"], self.distancia_pc)
            )

        with open("resultados/catalogo_completo.pkl", "rb") as archivo:
            datos = pickle.load(archivo)

        pkl_ok = (
            np.array_equal(datos["id"], self.id) and
            np.array_equal(datos["nombre"], self.nombre) and
            np.allclose(datos["temperatura_K"], self.temperatura_K) and
            np.allclose(datos["luminosidad_Lsol"], self.luminosidad_Lsol) and
            np.allclose(datos["distancia_pc"], self.distancia_pc)
        )

        print("Verificacion de archivos:")
        print("CSV:", csv_ok)
        print("NPY:", npy_ok)
        print("NPZ:", npz_ok)
        print("Pickle:", pkl_ok)

    def comparar_formatos(self):
        archivos = {
            "CSV": "resultados/catalogo_copia.csv",
            "NPY": "resultados/temperaturas.npy",
            "NPZ": "resultados/catalogo_completo.npz",
            "Pickle": "resultados/catalogo_completo.pkl"
        }

        resultados = []

        for formato, ruta in archivos.items():
            resultados.append({
                "Formato": formato,
                "Tamaño (bytes)": os.path.getsize(ruta)
            })

        return pd.DataFrame(resultados)

# Creamos el catálogo estelar y limpiamos sus datos automáticamente
catalogo = CatalogoEstelar("catalogo_estelar.csv")

Limpieza completada.
Estrellas validas: 92


In [3]:
catalogo.resumen()

Resumen del catalogo estelar
Cantidad de estrellas: 92
Temperatura media: 6388.6 K
Temperatura maxima: 9829.36 K
Luminosidad mediana: 6.03 Lsol
Distancia minima: 58.03 pc
Distancia maxima: 490.81 pc
Estrella mas caliente: Estrella_0006
Estrella mas cercana: Estrella_0039


In [4]:
catalogo.guardar_csv()
catalogo.guardar_npy()
catalogo.guardar_npz()
catalogo.guardar_pickle()

Catalogo guardado en CSV.
Temperaturas guardadas en NPY.
Catalogo guardado en NPZ.
Catalogo guardado en Pickle.


In [5]:
catalogo.verificar_archivos()

Limpieza completada.
Estrellas validas: 92
Verificacion de archivos:
CSV: True
NPY: True
NPZ: True
Pickle: True


In [6]:
catalogo.comparar_formatos()

,Formato,Tamaño (bytes)
0,CSV,6953
1,NPY,864
2,NPZ,3818
3,Pickle,4801


### (c) Comparación de formatos

El formato CSV permite almacenar el catálogo como texto organizado en columnas, por lo que es fácil de leer y compartir con otros usuarios. Sin embargo, puede ocupar más espacio y necesita convertirse a arreglos numéricos para realizar cálculos.

El formato NPY permite guardar arreglos de NumPy de forma eficiente, pero en esta tarea contiene únicamente las temperaturas, no el catálogo completo.

El formato NPZ permite guardar varios arreglos de NumPy en un solo archivo comprimido, por lo que resulta conveniente para almacenar el catálogo completo y trabajar con sus datos.

El formato Pickle permite guardar objetos de Python, como un diccionario con las columnas del catálogo. Es útil para recuperar estructuras de datos, aunque depende de Python y debe utilizarse con precaución cuando los archivos provienen de fuentes desconocidas.

Para compartir el catálogo con otros usuarios, elegiría **CSV**, porque es compatible con distintos programas y puede abrirse sin depender de Python. Para almacenar varios arreglos de NumPy en un único archivo, elegiría **NPZ**, porque permite agrupar los datos y recuperarlos posteriormente.